In [33]:
import pandas as pd
import numpy as np

In [34]:
df=pd.read_csv("Powerplant dataset.csv")

In [35]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9568 entries, 0 to 9567
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      9568 non-null   float64
 1   V       9568 non-null   float64
 2   AP      9568 non-null   float64
 3   RH      9568 non-null   float64
 4   PE      9568 non-null   float64
dtypes: float64(5)
memory usage: 373.9 KB


In [36]:
x=df.drop("PE",axis=1)
y=df["PE"]

In [37]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [38]:
# scaling
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [39]:
# converting to tensors
import torch
import torch.nn as nn
x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [40]:
# Dataset and dataloader
from torch.utils.data import TensorDataset,DataLoader
train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)


In [41]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

# Deep Learning 


In [42]:
# Define our ANN Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(x_train.shape[1], 6),
            nn.ReLU(),
    
            # 2nd hidden layer
            nn.Linear(6, 6),
            nn.ReLU(),
    
            # output layer
            nn.Linear(6, 1),
        )

    def forward(self, x):
        return self.model(x)

In [43]:
import torch.optim as optim
model=ANN()
# loss and optimizer
criterion=nn.MSELoss()
optimizer=optim.Adam(model.parameters())

In [44]:
# Train the ANN
train_losses = []
val_losses = []
best_val_loss = float("inf")

best_val_loss = float("inf")

epochs = 50

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # tot training loss for 1 epoch
    
    for xb, yb in train_loader:
        # xb = features of 1 batch
        # yb = labels of 1 batch
        optimizer.zero_grad()
        
        outputs = model(xb) # forward prop....predicted outputs for this batch
        loss = criterion(outputs, yb) # compute loss
        loss.backward() # back prop.. compute gradients
        optimizer.step() # params update
        
        running_loss += loss.item() # loss is a tensor => py float

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)


    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad(): # no gradients compute
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = criterion(outputs, yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if epoch_val_loss<best_val_loss:
        best_val_loss=epoch_val_loss
        torch.save(model.state_dict(),"best_loss.pt")

epoch 1/50 ==> train loss = 206845.56158854166 & val loss = 206251.59375
epoch 2/50 ==> train loss = 204419.96158854166 & val loss = 200204.5
epoch 3/50 ==> train loss = 190765.13313802084 & val loss = 177629.671875
epoch 4/50 ==> train loss = 159095.41507161458 & val loss = 137975.96875
epoch 5/50 ==> train loss = 115150.22737630208 & val loss = 92517.59375
epoch 6/50 ==> train loss = 73126.22125651041 & val loss = 56029.5078125
epoch 7/50 ==> train loss = 44050.28949381511 & val loss = 34165.52734375
epoch 8/50 ==> train loss = 28051.387638346354 & val loss = 22910.259765625
epoch 9/50 ==> train loss = 19859.95870768229 & val loss = 17105.7890625
epoch 10/50 ==> train loss = 15528.913783772787 & val loss = 13724.2060546875
epoch 11/50 ==> train loss = 12596.693385823568 & val loss = 11143.2978515625
epoch 12/50 ==> train loss = 10161.672377522786 & val loss = 8824.38671875
epoch 13/50 ==> train loss = 7954.540107218424 & val loss = 6722.74169921875
epoch 14/50 ==> train loss = 5959.7

In [45]:
# loading best model
model.load_state_dict(torch.load("best_loss.pt"))

<All keys matched successfully>

# Evaluation of model

In [47]:
model.eval()
with torch.no_grad():
    train_pred=model(x_train_tensor)
    test_pred=model(x_test_tensor)

    train_mse_loss=criterion(train_pred,y_train_tensor)
    test_mse_loss=criterion(test_pred,y_test_tensor)

print("Training mse loss",train_mse_loss.item())
print("Testing mse loss",test_mse_loss.item())

Training mse loss 20.440937042236328
Testing mse loss 18.499553680419922


In [48]:
# r2 score 
from sklearn.metrics import r2_score
r2=r2_score(test_pred,y_test_tensor)
print("r2_score:",r2)

r2_score: 0.9332789778709412


In [50]:
predicted_df=pd.DataFrame(test_pred.numpy(),columns=["Predicted_value"])
Actual_df=pd.DataFrame(y_test_tensor.numpy(),columns=["Actual_value"])
pd.concat([Actual_df,predicted_df],axis=1)

,Actual_value,Predicted_value
0,433.269989,434.504028
1,438.160004,435.713318
2,458.420013,461.052246
3,480.820007,476.539886
4,441.410004,433.823914
...,...,...
1909,456.700012,451.243835
1910,438.040009,428.652832
1911,467.799988,468.087952
1912,437.140015,431.955750
